## Sentence classification with BERT

### What is BERT?

BERT (Bidirectional Encoder Representations from Transformers), released in late 2018 by Google researchers is the model we’ll use to train our sentence classifier. We’ll use transfer learning on the pre-trained BERT model. This approach is better than training a deep model like GRU or LSTM from scratch as:

- the pre-trained BERT model weights already encode a lot of information about our language. As a result, it takes much less time to train our fine-tuned model. We’ll train our model with hyper-parameter sweeps to find the best combination within an hour on colab while training a single deep RNN model from scratch would take more than hundreds of hours on GPU!
- As we are essentially performing transfer learning, we need significantly less data to build an accurate system. Compared to millions of data points required to train a model from scratch, we can accomplish the same task with only a few thousand data points.

In [1]:
# If there's a GPU available...
import torch
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

No GPU available, using the CPU instead.


In [2]:
!pip install transformers

### The COLA dataset

We’ll use The Corpus of Linguistic Acceptability (CoLA) dataset for single sentence classification. It’s a set of sentences labeled as grammatically correct or incorrect. It was first published in May of 2018, and is one of the tests included in the “GLUE Benchmark” on which models like BERT are competing.

In [2]:
!pip install wget
import wget
import os

print('Downloading dataset...')

# The URL for the dataset zip file.
url = 'https://nyu-mll.github.io/CoLA/cola_public_1.1.zip'

# Download the file (if we haven't already)
if not os.path.exists('./cola_public_1.1.zip'):
    wget.download(url, './cola_public_1.1.zip')


if not os.path.exists('./cola_public/'):
    !unzip cola_public_1.1.zip


import pandas as pd

# Load the dataset into a pandas dataframe.
df = pd.read_csv("./cola_public/raw/in_domain_train.tsv", delimiter='\t', header=None, names=['sentence_source', 'label', 'label_notes', 'sentence'])

# Report the number of sentences.
print('Number of training sentences: {:,}\n'.format(df.shape[0]))

# Display 10 random rows from the data.
df.sample(10)

Defaulting to user installation because normal site-packages is not writeable
Number of training sentences: 8,551



,sentence_source,label,label_notes,sentence
5741,c_13,1,NaN,The puppy licked the kitten's face.
7427,sks13,0,*,"Yes, she used."
1809,r-67,1,NaN,Sonatas are difficult to play on this violin.
3958,ks08,1,NaN,Tom bathed Fido in the garage.
7860,ad03,1,NaN,Anson is incredibly difficult to please.
4041,ks08,1,NaN,Tom intends for Sam to review that book.
7231,sks13,1,NaN,Bill's mother's friends are waiting at the res...
7166,sks13,1,NaN,This girl in the red coat will eat her breakfa...
2700,l-93,1,NaN,Amanda drove the package to New York.
6366,d_98,0,*,Any lion is usually majestic.


In [3]:
# Get the lists of sentences and their labels.
sentences = df.sentence.values
labels = df.label.values

### Tokenization

As mentioned earlier, the sentences that are to be fed into the BERT model must be tokenized using the BERT tokenizer. Let’s take a look at an example.

In [4]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
'''
Original:  Our friends won't buy this analysis, let alone the next one we propose.
Tokenized:  ['our', 'friends', 'won', "'", 't', 'buy', 'this', 'analysis', ',', 'let', 'alone', 'the', 'next', 'one', 'we', 'propose', '.']
Token IDs:  [2256, 2814, 2180, 1005, 1056, 4965, 2023, 4106, 1010, 2292, 2894, 1996, 2279, 2028, 2057, 16599, 1012]
'''

Loading BERT tokenizer...


'\nOriginal:  Our friends won\'t buy this analysis, let alone the next one we propose.\nTokenized:  [\'our\', \'friends\', \'won\', "\'", \'t\', \'buy\', \'this\', \'analysis\', \',\', \'let\', \'alone\', \'the\', \'next\', \'one\', \'we\', \'propose\', \'.\']\nToken IDs:  [2256, 2814, 2180, 1005, 1056, 4965, 2023, 4106, 1010, 2292, 2894, 1996, 2279, 2028, 2057, 16599, 1012]\n'

Before we process the entire dataset using this tokenizer, there are a few conditions that we need to satisfy in order to setup the training data for BERT:

- Add special tokens to the start and end of each sentence. At the end of every sentence, we need to append the special [SEP] token and for classification tasks, we must prepend the special [CLS] token to the beginning of every sentence.
- Pad & truncate all sentences to a single constant length
- Explicitly differentiate real tokens from padding tokens with the “attention mask”. The “Attention Mask” is simply an array of 1s and 0s indicating which tokens are padding and which aren’t.

In [5]:
max_len = 0

# For every sentence...
for sent in sentences:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(sent, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

Max sentence length:  47


### Tonekize
To follow a general convention that the sizes should be in powers of 2, we’ll choose the closest number that is a power of 2, i.e, 64.

Now, we’re ready to perform the real tokenization. But as we’re using transformers, we can use an inbuilt function tokenizer.encode_plus which automates all of the following tasks:

1. Split the sentence into tokens.
2. Add the special `[CLS]` and `[SEP]` tokens.
3. Map the tokens to their IDs.
4. Pad or truncate all sentences to the same length.
5. Create the attention masks which explicitly differentiate real tokens from `[PAD]` tokens.

In [6]:
import torch
# Tokenize all of the sentences and map the tokens to thier word IDs.
input_ids = []
attention_masks = []

# For every sentence...
for sent in sentences:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer(
                        sent,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = 64,           # Pad & truncate all sentences. #next power of 2
                        padding='max_length',
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

# Print sentence 0, now as a list of IDs.
print('Original: ', sentences[0])
print('Token IDs:', input_ids[0])
print('Labels:', labels[0])
print('Attention Masks:', attention_masks[0])

Original:  Our friends won't buy this analysis, let alone the next one we propose.
Token IDs: tensor([  101,  2256,  2814,  2180,  1005,  1056,  4965,  2023,  4106,  1010,
         2292,  2894,  1996,  2279,  2028,  2057, 16599,  1012,   102,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])
Labels: tensor(1)
Attention Masks: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


### Train-Test Split

In [7]:
from torch.utils.data import TensorDataset, random_split

# Combine the training inputs into a TensorDataset.
dataset = TensorDataset(input_ids, attention_masks, labels)

# Create a 90-10 train-validation split.

# Calculate the number of samples to include in each set.
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

# Divide the dataset by randomly selecting samples.
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))

7,695 training samples
  856 validation samples


In [ ]:
!pip install datasets

In [8]:
from datasets import Dataset

In [9]:
# Create DataFrame

labels_list_tr=[train_dataset[i][2].item() for i in range(len(train_dataset))]
att_list_tr = [train_dataset[i][1].numpy() for i in range(len(train_dataset))]
ids_list_tr = [train_dataset[i][0].numpy() for i in range(len(train_dataset))]
labels_list_val=[val_dataset[i][2].item() for i in range(len(val_dataset))]
att_list_val = [val_dataset[i][1].numpy() for i in range(len(val_dataset))]
ids_list_val = [val_dataset[i][0].numpy() for i in range(len(val_dataset))]

traindata = pd.DataFrame({'input_ids': ids_list_tr, 'attention_mask': att_list_tr, 'label': labels_list_tr})
valdata = pd.DataFrame({'input_ids': ids_list_val, 'attention_mask': att_list_val, 'label': labels_list_val})
print(traindata.columns)
print(traindata.head())

# Convert the DataFrame into a Dataset
traindata = Dataset.from_pandas(traindata)
valdata = Dataset.from_pandas(valdata)

Index(['input_ids', 'attention_mask', 'label'], dtype='object')
                                           input_ids  \
0  [101, 2054, 2173, 2001, 1996, 2338, 2741, 2011...   
1  [101, 1996, 2214, 3899, 2351, 2112, 4576, 1012...   
2  [101, 2057, 10757, 2105, 1996, 2282, 1012, 102...   
3  [101, 2054, 2003, 2009, 3497, 2005, 7014, 2000...   
4  [101, 1045, 2113, 1045, 4149, 1996, 2338, 2008...   

                                      attention_mask  label  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, ...      0  
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, ...      0  
2  [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...      1  
3  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...      1  
4  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...      1  


### Fine-tune and evaluate Pre-trained BERT model

After we have processed our dataset, we can start training our model. We will use the bert-base-uncased model. The first step is to load our model with `BertForSequenceClassification` class from the [Hugging Face Hub](https://huggingface.co/bert-base-uncased). This will initialize the pre-trained BERT weights with a classification head on top. Here we pass the number of classes (2) from our dataset and the label names to have readable outputs for inference.

We have used `named_parameters()` function of the model to print out the model's parameters by names and dimensions of the weights for (1) the embedding layer; (2) the first of the twelve transformers, and (3) the output layer.

In [10]:
from transformers import BertForSequenceClassification

model = (BertForSequenceClassification.from_pretrained(
      "bert-base-uncased",
      num_labels = 2,
      output_attentions = False, # Whether the model returns attentions weights.
      output_hidden_states = False, # Whether the model returns all hidden-states.
  ).to(device))



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Get all of the model's parameters as a list of tuples.
params = list(model.named_parameters())

print('The BERT model has {:} different named parameters.\n'.format(len(params)))

print('==== Embedding Layer ====\n')

for p in params[0:5]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== First Transformer ====\n')

for p in params[5:21]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== Output Layer ====\n')

for p in params[-4:]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

The BERT model has 201 different named parameters.

==== Embedding Layer ====

bert.embeddings.word_embeddings.weight                  (30522, 768)
bert.embeddings.position_embeddings.weight                (512, 768)
bert.embeddings.token_type_embeddings.weight                (2, 768)
bert.embeddings.LayerNorm.weight                              (768,)
bert.embeddings.LayerNorm.bias                                (768,)

==== First Transformer ====

bert.encoder.layer.0.attention.self.query.weight          (768, 768)
bert.encoder.layer.0.attention.self.query.bias                (768,)
bert.encoder.layer.0.attention.self.key.weight            (768, 768)
bert.encoder.layer.0.attention.self.key.bias                  (768,)
bert.encoder.layer.0.attention.self.value.weight          (768, 768)
bert.encoder.layer.0.attention.self.value.bias                (768,)
bert.encoder.layer.0.attention.output.dense.weight        (768, 768)
bert.encoder.layer.0.attention.output.dense.bias              (

We evaluate our model during training. The `Trainer` supports evaluation during training by providing a `compute_metrics` method. We use the `evaluate` library to calculate the [f1 metric](https://huggingface.co/spaces/evaluate-metric/f1) during training on our test split.

In [12]:
from sklearn.metrics import f1_score
import numpy as np

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"f1": f1_score(labels, predictions, average = "weighted")}

The last step is to define the hyperparameters (`TrainingArguments`) we use for our training. Here we are passing the `torch_compile` option in the `TrainingArguments`.

We also leverage the [Hugging Face Hub](https://huggingface.co/models) integration of the `Trainer` to push our checkpoints, logs, and metrics during training into a repository.

In [13]:
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir="bert-base-uncased-finetuned-cola",
    num_train_epochs=3,             # number of training epochs
    learning_rate=2e-5,             # model learning rate
    per_device_train_batch_size=16, # batch size
    per_device_eval_batch_size=8,  # batch size
    eval_strategy="epoch",
    disable_tqdm=False,
    report_to="none",
    logging_steps=len(train_dataset)// 16,
    push_to_hub=False,
    log_level="error"
)

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=traindata,
    eval_dataset=valdata,
    compute_metrics=compute_metrics
)


In [14]:
# Start training
trainer.train()

c:\Users\cocod\Documents\repos\DataScienceStatlLearning\.conda\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [17]:
trainer.state.log_history

[{'loss': 0.4964,
  'grad_norm': 6.745864391326904,
  'learning_rate': 1.3347193347193348e-05,
  'epoch': 0.997920997920998,
  'step': 480},
 {'eval_loss': 0.44071418046951294,
  'eval_f1': 0.7759895316367742,
  'eval_runtime': 2.9972,
  'eval_samples_per_second': 285.603,
  'eval_steps_per_second': 35.7,
  'epoch': 1.0,
  'step': 481},
 {'loss': 0.2933,
  'grad_norm': 6.497178554534912,
  'learning_rate': 6.694386694386695e-06,
  'epoch': 1.995841995841996,
  'step': 960},
 {'eval_loss': 0.46143102645874023,
  'eval_f1': 0.8199485337012629,
  'eval_runtime': 3.0355,
  'eval_samples_per_second': 281.999,
  'eval_steps_per_second': 35.25,
  'epoch': 2.0,
  'step': 962},
 {'loss': 0.1855,
  'grad_norm': 1.504715919494629,
  'learning_rate': 4.158004158004159e-08,
  'epoch': 2.993762993762994,
  'step': 1440},
 {'eval_loss': 0.6313596963882446,
  'eval_f1': 0.8215141640637877,
  'eval_runtime': 3.1722,
  'eval_samples_per_second': 269.841,
  'eval_steps_per_second': 33.73,
  'epoch': 3.0,

In [20]:
# Prediction on validation dataset
val_pred_output = trainer.predict(valdata)
val_pred = np.argmax(val_pred_output.predictions, axis=1)
print('validation predictions: ', val_pred)
print('validation labels: ', val_pred_output.label_ids)

validation predictions:  [1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 0 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 1 1 1 0
 1 0 1 1 1 1 1 0 0 1 1 1 0 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 0 1 0 1 0 0
 1 1 1 0 1 1 0 0 1 1 0 1 1 1 0 1 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1 1 1 1 0 0 1
 1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1
 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 0 1 1 1 1 1
 0 1 1 1 1 1 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 1 0 0 1 1 1 1 1 1 1 0 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 0 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 0 0 0 1 1 1 0 0 1 0 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1
 1 1 0 0 1 0 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1
 1 0 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 0
 1 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1
 0 1 1 1 0 1 0 1 1 1 0 0 1 1 0 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 0 1 1 1 1 1
 0 1 1 1 1 1 1 1 1 0 0 0 1 0 1 1 0 1 1 1 0 1 1 0 0 0 1 0 1 1 1 0 1 1 1 1 1


In [29]:
from sklearn.metrics import accuracy_score
print(accuracy_score(val_pred_output.label_ids, val_pred))

0.8282710280373832


### Conclusion

This demonstrates that with a pre-trained BERT model you can quickly and effectively create a high quality model with minimal effort and training time using the pytorch interface, regardless of the specific NLP task you are interested in.



### References:
- https://github.com/BritneyMuller/colab-notebooks/blob/master/BERT.ipynb
- https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/pytorch-2-0-bert-text-classification.ipynb
- https://colab.research.google.com/github/DerwenAI/spaCy_tuTorial/blob/master/BERT_Fine_Tuning.ipynb#scrollTo=xlQG7qgkmf4n